<a href="https://colab.research.google.com/github/mar22266/PY2-DS/blob/Resultados-Parciales-y-Visualizaciones-Est%C3%A1ticas/PY2_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Proyecto 2 - Data Science - Modelo Final

## Integrantes

- Andre Marroquin - 22266
- Rodrigo Mansilla - 22611
- Sergio Orellana - 221122
- Carlos Valladares - 221164


# Proyecto 2 — Data Science

### link repo: https://github.com/mar22266/PY2-DS/tree/Resultados-Parciales-y-Visualizaciones-Est%C3%A1ticas

branch: Resultados Parciales y Visualizaciones Estáticas

## Fase 2: Investigación de modelos, selección, entrenamiento, evaluación y discusión

**Reto:** CGIAR – Ojos en el Terreno (detección de daños en cultivos)

**Objetivo:** Desarrollar un modelo multimodal de aprendizaje automático que integre información visual y metadatos para estimar el porcentaje de daño en cultivos y apoyar intervenciones agrícolas en regiones vulnerables al cambio climático.

**Objetivos específicos:**

- Diseñar e implementar modelos unimodales (visión y tabular) y multimodales (fusión tardía) que predigan el daño (extent) y evalúen su capacidad de generalización entre temporadas (validación LOSO).

- Comparar el desempeño de modelos clásicos (CML) y de aprendizaje profundo (DL) para determinar la arquitectura más adecuada para el contexto agrícola del reto.

- Visualizar e interpretar los resultados de los modelos para comprender los factores visuales y contextuales que contribuyen al daño, aportando evidencia para estrategias de manejo agrícola.

**Tarea de modelado.** Unidad de predicción: cada fila corresponde a un par (ID, DAMAGE) con metadatos (season, growth_stage, filename/ruta) y target EXTENT (0–100) en train; ausente en test.

**Objetivo:** predecir EXTENT para cada fila de test y formatear la salida según SampleSubmission.csv.

**Entradas:**

- Imagen: extracción de embeddings con un backbone preentrenado (p. ej., EfficientNet/ResNet) inicialmente congelado (fase de prototipo); se podrá descongelar parcialmente si el cómputo lo permite.

- Metadatos: season, growth_stage y otras variables no redundantes; excluir ID y columnas que induzcan fuga o proxy del label.

- Fusión: late fusion (concatenar embedding visual + metadatos procesados) → regresor (MLP ligero o GBDT tipo HistGradientBoosting/XGBoost).


## Configuración, imports y utilidades


In [ ]:
from __future__ import annotations
import os, time
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GroupKFold
import re
import warnings
from typing import Dict

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from sklearn.base import clone

from google.colab import drive

warnings.filterwarnings("ignore")


# Rutas
# DATA_DIR = Path(".")
# TRAIN_CSV = DATA_DIR / "Train.csv"
# TEST_CSV = DATA_DIR / "Test.csv"

# OUTPUT_DIR = Path("./outputs")
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# print("RUTAS OK:", TRAIN_CSV.exists(), TEST_CSV.exists(), SAMPLE_CSV.exists())

### Carga


In [11]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.bbox"] = "tight"

In [51]:
import os, time
import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')

BASE_PATH   = '/content/drive/MyDrive/PY2DS'
TRAIN_DIR_G = f'{BASE_PATH}/train'
TEST_DIR_G  = f'{BASE_PATH}/test'
TRAIN_CSV   = f'{BASE_PATH}/Train.csv'
TEST_CSV    = f'{BASE_PATH}/Test.csv'

LOCAL_TRAIN = '/content/train_images'
LOCAL_TEST  = '/content/test_images'

os.makedirs(LOCAL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_TEST, exist_ok=True)


print("Sincronizando TRAIN (solo faltantes)...")
start = time.time()
!rsync -a --ignore-existing --info=progress2 "{TRAIN_DIR_G}/" "{LOCAL_TRAIN}/"
print(f"   TRAIN sincronizado en {time.time()-start:.1f}s\n")

print("Sincronizando TEST (solo faltantes)...")
start = time.time()
!rsync -a --ignore-existing --info=progress2 "{TEST_DIR_G}/" "{LOCAL_TEST}/"
print(f"   TEST sincronizado en {time.time()-start:.1f}s\n")


train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

train_df['image_path'] = train_df['filename'].apply(lambda x: os.path.join(LOCAL_TRAIN, x))
test_df['image_path']  = test_df['filename'].apply(lambda x: os.path.join(LOCAL_TEST, x))

train_df = train_df[train_df['image_path'].apply(os.path.exists)].reset_index(drop=True)
test_df  = test_df[test_df['image_path'].apply(os.path.exists)].reset_index(drop=True)

print(f" Train válido: {len(train_df)} imágenes")
print(f" Test válido:  {len(test_df)} imágenes")

use_sample   = 'CONFIG' in globals() and CONFIG.get('use_sample', False)
sample_frac  = CONFIG.get('sample_frac', 0.3) if 'CONFIG' in globals() else 0.3
seed         = CONFIG.get('seed', 42) if 'CONFIG' in globals() else 42

if use_sample:
    print(f" Submuestreando {sample_frac*100:.0f}%...")
    train_df['extent_bin'] = pd.cut(train_df['extent'], bins=5, labels=False)
    train_df = (
        train_df
        .groupby('extent_bin', group_keys=False)
        .apply(lambda x: x.sample(frac=sample_frac, random_state=seed))
        .reset_index(drop=True)
    )
else:
    print(" Usando dataset completo.")

train_df['extent_bin'] = pd.cut(train_df['extent'], bins=5, labels=False)

train_subset, val_subset = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df['extent_bin'],
    random_state=seed
)

print(f" Train split: {len(train_subset)} | Val split: {len(val_subset)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sincronizando TRAIN (solo faltantes)...
              0   0%    0.00kB/s    0:00:00 (xfr#0, to-chk=0/26069)
   TRAIN sincronizado en 3.4s

Sincronizando TEST (solo faltantes)...
              0   0%    0.00kB/s    0:00:00 (xfr#0, to-chk=0/8664)
   TEST sincronizado en 1.1s

Train shape: (26068, 6), Test shape: (8663, 5)
 Train válido: 26068 imágenes
 Test válido:  8663 imágenes
 Usando dataset completo.
 Train split: 20854 | Val split: 5214


## Carga y verificación de datos


In [ ]:
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
print("Train")
print(train.head(3))
print("Test")
print(test.head(3))
print("shapes:", train.shape, test.shape)

Train
              ID                        filename growth_stage damage  extent  \
0  ID_1S8OOWQYCB  L427F01330C01S03961Rp02052.jpg            S     WD       0   
1  ID_0MD959MIZ0      L1083F00930C39S12674Ip.jpg            V      G       0   
2  ID_JRJCI4Q11V      24_initial_1_1463_1463.JPG            V      G       0   

   season  
0  SR2020  
1  SR2021  
2  LR2020  
Test
              ID                         filename growth_stage damage  season
0  ID_ROOWKB90UZ   L122F00315C01S02151Rp04021.jpg            V     WD  SR2020
1  ID_PTEDRY0CYM  L1089F03254C01S08845Rp25119.jpg            F     WD  LR2021
2  ID_5WJXDV96R4   L365F01913C39S12578Rp42918.jpg            V     WD  SR2021
shapes: (26068, 6) (8663, 5)


## Preprocesamiento columnas, mapeos, features


In [ ]:
def normalize_season(s: str) -> str:
    if pd.isna(s):
        return "UNKNOWN"
    t = str(s).upper().strip().replace(" ", "")
    t = re.sub(r"[^A-Z0-9]", "", t)
    return t


_STAGE_MAP = {
    "F": "F",
    "FLOWERING": "F",
    "M": "M",
    "MATURITY": "M",
    "S": "S",
    "SOWING": "S",
    "V": "V",
    "VEGETATIVE": "V",
}


def normalize_stage(s: str) -> str:
    if pd.isna(s):
        return "UNK"
    t = str(s).upper().strip()
    return _STAGE_MAP.get(t, t if t in {"F", "M", "S", "V"} else "UNK")


def filename_features(fname: str) -> Dict[str, int]:
    d = {"has_jpg": 0, "has_jpeg": 0, "len_name": 0, "digits_sum": 0}
    if not isinstance(fname, str):
        return d
    name = fname.strip()
    d["has_jpg"] = int(name.lower().endswith(".jpg"))
    d["has_jpeg"] = int(name.lower().endswith(".jpeg"))
    d["len_name"] = len(name)
    digits = re.findall(r"\d+", name)
    d["digits_sum"] = int(np.sum([int(x) for x in digits])) if digits else 0
    return d


def apply_preprocessing(df: pd.DataFrame, is_train: bool) -> pd.DataFrame:
    out = df.copy()
    out.columns = [c.lower() for c in out.columns]
    expected = {"id", "filename", "growth_stage", "damage", "season"}
    if not expected.issubset(out.columns):
        raise ValueError(f"Faltan columnas requeridas: {expected - set(out.columns)}")

    out["season"] = out["season"].map(normalize_season)
    out["stage_code"] = out["growth_stage"].map(normalize_stage)
    out["damage"] = out["damage"].astype(str).str.upper().str.strip()

    if "extent" in out.columns:
        out["extent"] = (
            pd.to_numeric(out["extent"], errors="coerce").fillna(0.0).clip(0, 100)
        )

    feats = out["filename"].apply(filename_features).apply(pd.Series)
    out = pd.concat([out, feats], axis=1)

    keep_cols = [
        "id",
        "damage",
        "season",
        "stage_code",
        "has_jpg",
        "has_jpeg",
        "len_name",
        "digits_sum",
    ]
    if "extent" in out.columns:
        keep_cols.append("extent")

    return out[keep_cols]

## Estrategia MultiModal

MobileNetV3-small embeddings (1280D) + XGBoost


In [54]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [ ]:
import torchvision.models as torch_models
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMG_SIZE = 224
img_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)


class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.paths = df["image_path"].values
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img


weights = torch_models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
backbone = torch_models.mobilenet_v3_small(weights=weights)
backbone.classifier = nn.Identity()
backbone.eval()
backbone.to(device)


def extract_embeddings(
    df: pd.DataFrame, batch_size: int = 64, desc: str = ""
) -> np.ndarray:
    dataset = ImageDataset(df, transform=img_transform)
    loader = DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )
    feats = []
    with torch.no_grad():
        for imgs in tqdm(loader, desc=desc, total=len(loader)):
            imgs = imgs.to(device)
            emb = backbone(imgs)
            feats.append(emb.cpu().numpy())
    return np.vstack(feats)


print("\nExtrayendo embeddings MobileNetV3-small...")
emb_train = extract_embeddings(train_subset, batch_size=64, desc="Train embeddings")
emb_val = extract_embeddings(val_subset, batch_size=64, desc="Val embeddings")
emb_test = extract_embeddings(test_df, batch_size=64, desc="Test embeddings")

print("Shapes embeddings:")
print("  emb_train:", emb_train.shape)
print("  emb_val  :", emb_val.shape)
print("  emb_test :", emb_test.shape)

assert emb_train.shape[0] == X_meta_train.shape[0] == y_train.shape[0]
assert emb_val.shape[0] == X_meta_val.shape[0] == y_val.shape[0]

Device: cpu

Extrayendo embeddings MobileNetV3-small...


Train embeddings:   0%|          | 0/326 [00:00<?, ?it/s]

Val embeddings:   0%|          | 0/82 [00:00<?, ?it/s]

Test embeddings:   0%|          | 0/136 [00:00<?, ?it/s]

Shapes embeddings:
  emb_train: (20854, 576)
  emb_val  : (5214, 576)
  emb_test : (8663, 576)


### PCA + scaling + fusión temprana


In [ ]:
VIS_N_COMP = 256
VIS_WEIGHT = 0.3

X_meta_tr = np.asarray(X_meta_train)
X_meta_va = np.asarray(X_meta_val)
X_meta_te = np.asarray(X_meta_test)

X_vis_tr = np.asarray(emb_train, dtype=np.float32)
X_vis_va = np.asarray(emb_val, dtype=np.float32)
X_vis_te = np.asarray(emb_test, dtype=np.float32)

scaler_vis_raw = StandardScaler()
X_vis_tr_std = scaler_vis_raw.fit_transform(X_vis_tr)
X_vis_va_std = scaler_vis_raw.transform(X_vis_va)
X_vis_te_std = scaler_vis_raw.transform(X_vis_te)

pca_vis = PCA(n_components=VIS_N_COMP, random_state=RANDOM_SEED)
X_vis_tr_pca = pca_vis.fit_transform(X_vis_tr_std)
X_vis_va_pca = pca_vis.transform(X_vis_va_std)
X_vis_te_pca = pca_vis.transform(X_vis_te_std)

print(f"\nEmbeddings reducidos: {VIS_N_COMP} componentes")
print(f"Varianza explicada: {pca_vis.explained_variance_ratio_.sum()*100:.2f}%")

scaler_meta = StandardScaler()
scaler_vis = StandardScaler()

X_meta_tr_std = scaler_meta.fit_transform(X_meta_tr)
X_meta_va_std = scaler_meta.transform(X_meta_va)
X_meta_te_std = scaler_meta.transform(X_meta_te)

X_vis_tr_pca_std = scaler_vis.fit_transform(X_vis_tr_pca)
X_vis_va_pca_std = scaler_vis.transform(X_vis_va_pca)
X_vis_te_pca_std = scaler_vis.transform(X_vis_te_pca)


def fuse(meta_std, vis_pca_std, alpha=VIS_WEIGHT):
    return np.hstack([meta_std, alpha * vis_pca_std])


X_fus_tr = fuse(X_meta_tr_std, X_vis_tr_pca_std)
X_fus_va = fuse(X_meta_va_std, X_vis_va_pca_std)
X_fus_te = fuse(X_meta_te_std, X_vis_te_pca_std)

print(
    "Shapes fusión:",
    "Train:",
    X_fus_tr.shape,
    "Val:",
    X_fus_va.shape,
    "Test:",
    X_fus_te.shape,
)


Embeddings reducidos: 256 componentes
Varianza explicada: 93.02%
Shapes fusión: Train: (20854, 276) Val: (5214, 276) Test: (8663, 276)


### Modelo base tabular (RF) y predicciones base


In [ ]:
rf_meta = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=1,
    min_samples_split=2,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

rf_meta.fit(X_meta_tr, y_train)
base_tr = rf_meta.predict(X_meta_tr)
base_va = rf_meta.predict(X_meta_va)

print("\nBaseline tabular (RF metadatos):")
metrics_rf_tr = eval_metrics(y_train, base_tr, label="TRAIN RF")
metrics_rf_va = eval_metrics(y_val, base_va, label="VAL   RF")


Baseline tabular (RF metadatos):
TRAIN RF -> MAE=2.2749 | RMSE=6.8683 | R2=0.8638
VAL   RF -> MAE=3.0289 | RMSE=9.3437 | R2=0.7481


### Estrategia A: Early Fusion directo (XGB sobre features fusionadas)


In [ ]:
import xgboost as xgb
from xgboost.callback import TrainingCallback
from tqdm.auto import tqdm


class TQDMCallback(TrainingCallback):
    def __init__(self, total: int, desc: str):
        self.total = total
        self.desc = desc
        self.pbar = None

    def before_training(self, model):
        self.pbar = tqdm(total=self.total, desc=self.desc, leave=True)
        return model

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False

    def after_training(self, model):
        if self.pbar is not None:
            self.pbar.close()
        return model


def train_xgb_with_progress(desc, params, dtrain, num_boost_round, evals):
    cb = TQDMCallback(total=num_boost_round, desc=desc)
    bst = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=evals,
        verbose_eval=False,
        callbacks=[cb],
    )
    return bst

In [ ]:
import xgboost as xgb
from tqdm.auto import tqdm
from xgboost.callback import TrainingCallback

dtrain_fus = xgb.DMatrix(X_fus_tr, label=y_train)
dval_fus = xgb.DMatrix(X_fus_va, label=y_val)

params_fusion = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "seed": RANDOM_SEED,
}

num_boost_round = 500

cb = TQDMCallback(total=num_boost_round)

bst_fusion = xgb.train(
    params=params_fusion,
    dtrain=dtrain_fus,
    num_boost_round=num_boost_round,
    evals=[(dtrain_fus, "train"), (dval_fus, "val")],
    verbose_eval=False,
    callbacks=[cb],
)

y_va_fus = bst_fusion.predict(dval_fus)
metrics_fusion = eval_metrics(y_val, y_va_fus, label="VAL Early Fusion (directo)")

XGB Early Fusion:   0%|          | 0/500 [00:00<?, ?it/s]

VAL Early Fusion (directo) -> MAE=2.5629 | RMSE=7.2624 | R2=0.8478


### Estrategia B: Residual learning (RF + XGB pequeño sobre residuales)


In [ ]:
# Residuos a corregir por la visión
res_tr = y_train - base_tr
res_va = y_val - base_va

dtrain_res = xgb.DMatrix(X_fus_tr, label=res_tr)
dval_res = xgb.DMatrix(X_fus_va, label=res_va)

params_residual = {
    "objective": "reg:squarederror",
    "max_depth": 4,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "seed": RANDOM_SEED,
}

num_boost_round_res = 300

bst_residual = train_xgb_with_progress(
    desc="XGB Residual",
    params=params_residual,
    dtrain=dtrain_res,
    num_boost_round=num_boost_round_res,
    evals=[(dtrain_res, "train"), (dval_res, "val")],
)

res_pred_va = bst_residual.predict(dval_res)
y_va_residual = base_va + res_pred_va
metrics_residual = eval_metrics(y_val, y_va_residual, label="VAL Residual RF + Visual")

XGB Residual:   0%|          | 0/300 [00:00<?, ?it/s]

VAL Residual RF + Visual -> MAE=2.7703 | RMSE=8.3347 | R2=0.7996


###


In [ ]:
def better(m1, m2, key="MAE"):
    return m1[key] < m2[key]


print("\nComparación en VALIDACIÓN:")
print("  Early Fusion directo:", metrics_fusion)
print("  Residual RF+Visual   :", metrics_residual)

if better(metrics_residual, metrics_fusion, "MAE"):
    best_strategy = "residual"
    print("\n=> Estrategia elegida: Residual learning (RF + XGB pequeño)")
else:
    best_strategy = "fusion_directa"
    print("\n=> Estrategia elegida: Early Fusion directo")


Comparación en VALIDACIÓN:
  Early Fusion directo: {'MAE': 2.562878370285034, 'RMSE': np.float64(7.262360132548111), 'R2': 0.847844123840332}
  Residual RF+Visual   : {'MAE': 2.7703201124877013, 'RMSE': np.float64(8.334658338947408), 'R2': 0.7995949135211533}

=> Estrategia elegida: Early Fusion directo


In [ ]:
import itertools, random

param_space = {
    "max_depth": [4, 5, 6],
    "min_child_weight": [1, 5, 10],
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
}

candidates = list(
    itertools.product(
        param_space["max_depth"],
        param_space["min_child_weight"],
        param_space["reg_lambda"],
    )
)
random.shuffle(candidates)

best_mae = np.inf
best_params = None
best_model = None

for i, (md, mcw, rl) in enumerate(candidates[:15], start=1):
    params = {
        "objective": "reg:squarederror",
        "max_depth": md,
        "eta": 0.05,
        "min_child_weight": mcw,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": rl,
        "tree_method": "hist",
        "eval_metric": "rmse",
        "seed": RANDOM_SEED,
    }
    print(f"\n[{i}] max_depth={md}, min_child_weight={mcw}, reg_lambda={rl}")
    bst_tmp = train_xgb_with_progress(
        desc=f"XGB Early Fusion [{i}]",
        params=params,
        dtrain=dtrain_fus,
        num_boost_round=500,
        evals=[(dtrain_fus, "train"), (dval_fus, "val")],
    )
    y_va_tmp = bst_tmp.predict(dval_fus)
    m = eval_metrics(y_val, y_va_tmp)
    if m["MAE"] < best_mae:
        best_mae = m["MAE"]
        best_params = params
        best_model = bst_tmp

print("\nMejores params:", best_params)
print("Mejor MAE val:", best_mae)


[1] max_depth=6, min_child_weight=5, reg_lambda=0.0


XGB Early Fusion [1]:   0%|          | 0/500 [00:00<?, ?it/s]


[2] max_depth=5, min_child_weight=1, reg_lambda=0.0


XGB Early Fusion [2]:   0%|          | 0/500 [00:00<?, ?it/s]


[3] max_depth=4, min_child_weight=10, reg_lambda=10.0


...


[14] max_depth=4, min_child_weight=5, reg_lambda=10.0


XGB Early Fusion [14]:   0%|          | 0/500 [00:00<?, ?it/s]


[15] max_depth=6, min_child_weight=10, reg_lambda=0.0


XGB Early Fusion [15]:   0%|          | 0/500 [00:00<?, ?it/s]


Mejores params: {'objective': 'reg:squarederror', 'max_depth': 6, 'eta': 0.05, 'min_child_weight': 5, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 5.0, 'tree_method': 'hist', 'eval_metric': 'rmse', 'seed': 42}
Mejor MAE val: 2.5671544075012207


In [ ]:
# 1. Fijar mejores parámetros de fusión
params_fusion_final = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "eta": 0.05,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 5.0,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "seed": RANDOM_SEED,
}

num_boost_round_fus = 500


def fuse(meta_std, vis_pca_std, alpha):
    return np.hstack([meta_std, alpha * vis_pca_std])


alphas = [0.2, 0.3, 0.4]

best_alpha = None
best_mae_alpha = np.inf
best_model_alpha = None

for alpha in alphas:
    print(f"\nProbando VIS_WEIGHT = {alpha}")

    # Re-fusión con ese alpha
    X_fus_tr = fuse(X_meta_tr_std, X_vis_tr_pca_std, alpha)
    X_fus_va = fuse(X_meta_va_std, X_vis_va_pca_std, alpha)

    dtrain_fus = xgb.DMatrix(X_fus_tr, label=y_train)
    dval_fus = xgb.DMatrix(X_fus_va, label=y_val)

    bst_fus = train_xgb_with_progress(
        desc=f"XGB Early Fusion (alpha={alpha})",
        params=params_fusion_final,
        dtrain=dtrain_fus,
        num_boost_round=num_boost_round_fus,
        evals=[(dtrain_fus, "train"), (dval_fus, "val")],
    )

    y_va_fus = bst_fus.predict(dval_fus)
    m = eval_metrics(y_val, y_va_fus, label=f"VAL Early Fusion (alpha={alpha})")

    if m["MAE"] < best_mae_alpha:
        best_mae_alpha = m["MAE"]
        best_alpha = alpha
        best_model_alpha = bst_fus

print("\n>>> Mejor VIS_WEIGHT (alpha):", best_alpha)
print(">>> Mejor MAE val:", best_mae_alpha)


=== Probando VIS_WEIGHT = 0.2 ===


XGB Early Fusion (alpha=0.2):   0%|          | 0/500 [00:00<?, ?it/s]

VAL Early Fusion (alpha=0.2) -> MAE=2.5670 | RMSE=7.2902 | R2=0.8467

=== Probando VIS_WEIGHT = 0.3 ===


XGB Early Fusion (alpha=0.3):   0%|          | 0/500 [00:00<?, ?it/s]

VAL Early Fusion (alpha=0.3) -> MAE=2.5672 | RMSE=7.2910 | R2=0.8466

=== Probando VIS_WEIGHT = 0.4 ===


XGB Early Fusion (alpha=0.4):   0%|          | 0/500 [00:00<?, ?it/s]

VAL Early Fusion (alpha=0.4) -> MAE=2.5670 | RMSE=7.2902 | R2=0.8467

>>> Mejor VIS_WEIGHT (alpha): 0.2
>>> Mejor MAE val: 2.5670104026794434


### Entrenamiento final (train + val)


In [ ]:
best_alpha = 0.2

X_meta_full = np.vstack([X_meta_tr, X_meta_va])
X_vis_full = np.vstack([X_vis_tr, X_vis_va])
y_full = np.concatenate([y_train, y_val])

scaler_vis_raw_full = StandardScaler()
X_vis_full_std = scaler_vis_raw_full.fit_transform(X_vis_full)
X_vis_te_std = scaler_vis_raw_full.transform(X_vis_te)

pca_vis_full = PCA(n_components=VIS_N_COMP, random_state=RANDOM_SEED)
X_vis_full_pca = pca_vis_full.fit_transform(X_vis_full_std)
X_vis_te_pca = pca_vis_full.transform(X_vis_te_std)

scaler_meta_full = StandardScaler()
X_meta_full_std = scaler_meta_full.fit_transform(X_meta_full)
X_meta_te_std = scaler_meta_full.transform(X_meta_te)

scaler_vis_pca_full = StandardScaler()
X_vis_full_pca_std = scaler_vis_pca_full.fit_transform(X_vis_full_pca)
X_vis_te_pca_std = scaler_vis_pca_full.transform(X_vis_te_pca)


def fuse(meta_std, vis_pca_std, alpha):
    return np.hstack([meta_std, alpha * vis_pca_std])


X_fus_full = fuse(X_meta_full_std, X_vis_full_pca_std, best_alpha)
X_fus_te = fuse(X_meta_te_std, X_vis_te_pca_std, best_alpha)

print("Full fusion shapes:", X_fus_full.shape, X_fus_te.shape)

# 3) DMatrix para XGBoost
dtrain_full = xgb.DMatrix(X_fus_full, label=y_full)
dtest_fus = xgb.DMatrix(X_fus_te)

# 4) Mejores hiperparámetros encontrados
params_fusion_final = {
    "objective": "reg:squarederror",
    "max_depth": 6,
    "eta": 0.05,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 5.0,
    "tree_method": "hist",
    "eval_metric": "rmse",
    "seed": RANDOM_SEED,
}

num_boost_round_fus = 500

bst_fusion_final = train_xgb_with_progress(
    desc="XGB Early Fusion FINAL",
    params=params_fusion_final,
    dtrain=dtrain_full,
    num_boost_round=num_boost_round_fus,
    evals=[(dtrain_full, "train")],
)

y_test_pred = bst_fusion_final.predict(dtest_fus)
y_test_pred_clip = clip_and_round(y_test_pred)

print("\nPredicciones multimodales en TEST (FINAL):")
print(
    f"min={y_test_pred_clip.min():.2f}, "
    f"max={y_test_pred_clip.max():.2f}, "
    f"mean={y_test_pred_clip.mean():.2f}, "
    f"std={y_test_pred_clip.std():.2f}"
)

submission_df = test_clean[["id", "damage"]].copy()
submission_df["extent"] = y_test_pred_clip.astype(int)

print("\nEjemplo submission_df FINAL:")
print(submission_df.head())

Full fusion shapes: (26068, 276) (8663, 276)


XGB Early Fusion FINAL:   0%|          | 0/500 [00:00<?, ?it/s]


Predicciones multimodales en TEST (FINAL):
min=0.00, max=95.00, mean=7.15, std=16.81

Ejemplo submission_df FINAL:
              id damage  extent
0  ID_ROOWKB90UZ     WD       0
1  ID_PTEDRY0CYM     WD       0
2  ID_5WJXDV96R4     WD       0
3  ID_DM4AQLXXYG      G       0
4  ID_V6YTIT7I2S      G       0


In [ ]:
y_test_pred_clip = clip_and_round(y_test_pred)

print("\nPredicciones multimodales en TEST:")
print(
    f"min={y_test_pred_clip.min():.2f}, "
    f"max={y_test_pred_clip.max():.2f}, "
    f"mean={y_test_pred_clip.mean():.2f}, "
    f"std={y_test_pred_clip.std():.2f}"
)

submission_df = test_clean[["id", "damage"]].copy()
submission_df["extent"] = y_test_pred_clip.astype(int)

print("\nEjemplo de submission_df:")
print(submission_df.head())


Predicciones multimodales en TEST:
min=0.00, max=95.00, mean=7.15, std=16.81

Ejemplo de submission_df:
              id damage  extent
0  ID_ROOWKB90UZ     WD       0
1  ID_PTEDRY0CYM     WD       0
2  ID_5WJXDV96R4     WD       0
3  ID_DM4AQLXXYG      G       0
4  ID_V6YTIT7I2S      G       0


In [ ]:
# Distribución de y en train_full
print("Train y_full:")
print(
    f"min={y_full.min():.2f}, max={y_full.max():.2f}, mean={y_full.mean():.2f}, std={y_full.std():.2f}"
)

# Distribución de predicciones en val (ya la tienes, pero recalculamos para el modelo final si quieres)
dval_full = xgb.DMatrix(X_fus_va)
y_val_pred_final = bst_fusion_final.predict(dval_full)
print("Val pred final:")
print(
    f"min={y_val_pred_final.min():.2f}, max={y_val_pred_final.max():.2f}, "
    f"mean={y_val_pred_final.mean():.2f}, std={y_val_pred_final.std():.2f}"
)

Train y_full:
min=0.00, max=100.00, mean=7.10, std=18.61
Val pred final:
min=-3.31, max=74.57, mean=10.08, std=14.36


In [76]:
import os, json, joblib
import xgboost as xgb

ARTIFACT_DIR = "/content/artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# 1) Preprocesador tabular (metadatos)
joblib.dump(pre_final, os.path.join(ARTIFACT_DIR, "pre_final.pkl"))

# 2) Scalers + PCA para rama visual y tabular
joblib.dump(scaler_vis_raw_full, os.path.join(ARTIFACT_DIR, "scaler_vis_raw_full.pkl"))
joblib.dump(pca_vis_full,        os.path.join(ARTIFACT_DIR, "pca_vis_full.pkl"))
joblib.dump(scaler_meta_full,    os.path.join(ARTIFACT_DIR, "scaler_meta_full.pkl"))
joblib.dump(scaler_vis_pca_full, os.path.join(ARTIFACT_DIR, "scaler_vis_pca_full.pkl"))

# 3) Modelo XGBoost multimodal final
bst_fusion_final.save_model(os.path.join(ARTIFACT_DIR, "xgb_early_fusion_final.json"))

# 4) Modelo RF baseline tabular (opcional)
joblib.dump(rf_meta, os.path.join(ARTIFACT_DIR, "rf_meta_baseline.pkl"))

# 5) Config mínima para la app
config = {
    "VIS_N_COMP": int(VIS_N_COMP),
    "VIS_WEIGHT": float(best_alpha),
    "RANDOM_SEED": int(RANDOM_SEED),
    "cat_cols": ["damage", "season", "stage_code"],
    "num_cols": ["has_jpg", "has_jpeg", "len_name", "digits_sum"],
}
with open(os.path.join(ARTIFACT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("Artefactos guardados en:", ARTIFACT_DIR)
!ls -lh "{ARTIFACT_DIR}"


Artefactos guardados en: /content/artifacts
total 35M
-rw-r--r-- 1 root root  216 Nov 17 02:29 config.json
-rw-r--r-- 1 root root 583K Nov 17 02:29 pca_vis_full.pkl
-rw-r--r-- 1 root root 3.2K Nov 17 02:29 pre_final.pkl
-rw-r--r-- 1 root root  31M Nov 17 02:29 rf_meta_baseline.pkl
-rw-r--r-- 1 root root 1.1K Nov 17 02:29 scaler_meta_full.pkl
-rw-r--r-- 1 root root 6.7K Nov 17 02:29 scaler_vis_pca_full.pkl
-rw-r--r-- 1 root root  15K Nov 17 02:29 scaler_vis_raw_full.pkl
-rw-r--r-- 1 root root 3.2M Nov 17 02:29 xgb_early_fusion_final.json


In [ ]:
from google.colab import files

for fname in [
    "pre_final.pkl",
    "scaler_vis_raw_full.pkl",
    "pca_vis_full.pkl",
    "scaler_meta_full.pkl",
    "scaler_vis_pca_full.pkl",
    "xgb_early_fusion_final.json",
    "rf_meta_baseline.pkl",
    "config.json",
]:
    files.download(os.path.join(ARTIFACT_DIR, fname))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>